In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, IntegerType, DoubleType
from pyspark.sql.window import Window
from pyspark.sql.functions import to_timestamp
from datetime import date
import calendar

In [0]:
# Cargar datos de la tabla bronze filtrando por el último data_ingestion_ts
df_bronze = spark.table("adbsmartdatamanuelestrada.bronze.base_produccion_brz")

# Obtener el último data_ingestion_ts
max_ts = df_bronze.agg(F.max("data_ingestion_ts").alias("max_ts")).collect()[0]["max_ts"]

# Filtrar solo los registros del último data_ingestion_ts
df_base = df_bronze.filter(F.col("data_ingestion_ts") == max_ts)

print(f"Último data_ingestion_ts: {max_ts}")
print(f"Total registros cargados: {df_base.count()}")

display(df_base.limit(5))

In [0]:
# ============================================================
# VALIDACIÓN #1: CONTROL DE DUPLICADOS POR MES (FAIL FAST)
# ============================================================

print("=" * 60)
print("VALIDACIÓN #1: CONTROL DE DUPLICADOS POR MES")
print("=" * 60)

# 1. Extraer el mes de los datos cargados desde Bronze
# Primero convertimos fecha_orden (string) a Date para poder extraer el mes
df_temp_mes = df_base.withColumn(
    "fecha_parsed_temp",
    F.to_date(F.col("fecha_orden"), "d/MM/yyyy")
).withColumn(
    "mes_bronze",
    F.concat(
        F.year("fecha_parsed_temp"),
        F.lpad(F.month("fecha_parsed_temp"), 2, "0")
    )
)

# Obtener los meses únicos en los datos de Bronze
meses_en_bronze = df_temp_mes.select("mes_bronze").distinct().collect()
meses_detectados = [row['mes_bronze'] for row in meses_en_bronze if row['mes_bronze'] is not None]

if not meses_detectados:
    raise ValueError(
        f"❌ ERROR: No se pudo extraer el mes de los datos de Bronze.\n"
        f"   Verifique que fecha_orden tenga valores válidos."
    )

print(f"\nMes(es) detectado(s) en Bronze: {', '.join(meses_detectados)}")

# Validar que solo haya UN mes en los datos
if len(meses_detectados) > 1:
    raise ValueError(
        f"❌ ERROR: Los datos de Bronze contienen múltiples meses: {', '.join(meses_detectados)}\n"
        f"   El procesamiento debe ser de un solo mes a la vez."
    )

mes_a_procesar = meses_detectados[0]
print(f"Mes a procesar: {mes_a_procesar}")

# 2. Verificar si el mes YA existe en la tabla Silver
table_name = "adbsmartdatamanuelestrada.silver.quality_layer_slv"

try:
    # Intentar leer la tabla Silver existente
    tabla_silver = spark.table(table_name)
    
    # Extraer meses existentes en Silver
    meses_silver = tabla_silver.select(
        F.concat(
            F.year("fecha_orden"),
            F.lpad(F.month("fecha_orden"), 2, "0")
        ).alias("mes_silver")
    ).distinct().collect()
    
    meses_en_silver = [row['mes_silver'] for row in meses_silver]
    print(f"\nMeses ya existentes en Silver: {', '.join(meses_en_silver)}")
    
    # ❌ FALLA si el mes ya existe
    if mes_a_procesar in meses_en_silver:
        raise ValueError(
            f"❌ ERROR: El mes {mes_a_procesar} ya existe en la tabla Silver.\n"
            f"   No se procesará para evitar duplicados.\n"
            f"   Para reprocesar, primero elimine el mes de Silver:\n"
            f"   DELETE FROM {table_name} \n"
            f"   WHERE CONCAT(YEAR(fecha_orden), LPAD(MONTH(fecha_orden), 2, '0')) = '{mes_a_procesar}'"
        )
    
    print(f"\n✅ El mes {mes_a_procesar} NO existe en Silver.")
    print(f"   Se procederá con el procesamiento.")
    
except Exception as e:
    # Si la tabla no existe, es la primera carga
    if "Table or view not found" in str(e) or "cannot be found" in str(e).lower():
        print(f"\nℹ️  La tabla Silver no existe aún.")
        print(f"   Se creará con los datos del mes {mes_a_procesar}.")
    else:
        # Si es otro error (como la validación que falló), re-raise
        raise

print("=" * 60)

In [0]:
# ============================================================
# VALIDACIÓN #2: TIPO DE CAMBIO DISPONIBLE (FAIL FAST)
# ============================================================

print("=" * 60)
print("VALIDACIÓN #2: TIPO DE CAMBIO DISPONIBLE")
print("=" * 60)

# 1. Convertir mes_a_procesar (YYYYMM) a formato DATE (primer día del mes)
year_mes = int(mes_a_procesar[:4])
month_mes = int(mes_a_procesar[4:6])
fecha_mes = date(year_mes, month_mes, 1)

print(f"\nMes a procesar: {mes_a_procesar}")
print(f"Fecha equivalente: {fecha_mes}")

# 2. Verificar si existe el tipo de cambio para ese mes
tabla_tipo_cambio = "adbsmartdatamanuelestrada.silver.tipo_cambio_bcrp_slv"

try:
    df_tipo_cambio = spark.table(tabla_tipo_cambio)
    
    # Buscar el mes específico
    tipo_cambio_mes = df_tipo_cambio.filter(
        F.col("mes") == F.lit(fecha_mes)
    ).collect()
    
    if len(tipo_cambio_mes) == 0:
        # El mes NO tiene tipo de cambio - DETENER
        meses_disponibles = df_tipo_cambio.select(
            F.min("mes").alias("mes_min"),
            F.max("mes").alias("mes_max")
        ).collect()[0]
        
        mes_min = meses_disponibles["mes_min"]
        mes_max = meses_disponibles["mes_max"]
        
        raise ValueError(
            f"❌ ERROR: El mes {mes_a_procesar} ({fecha_mes}) NO tiene tipo de cambio disponible.\n"
            f"\n"
            f"   Rango disponible: {mes_min} hasta {mes_max}\n"
            f"\n"
            f"   SOLUCIÓN:\n"
            f"   1. Ejecuta el notebook 'str_gld_tipoCambioBCRP'\n"
            f"   2. Verifica que el BCRP haya publicado el tipo de cambio para {mes_a_procesar}\n"
        )
    
    # El mes SÍ tiene tipo de cambio - CONTINUAR
    valor_tipo_cambio = tipo_cambio_mes[0]["valor"]
    
    print(f"\n✅ Tipo de cambio DISPONIBLE para {mes_a_procesar}")
    print(f"   Valor: S/. {valor_tipo_cambio:.4f}")
    print(f"   Se procederá con el procesamiento.")
    
except Exception as e:
    if "Table or view not found" in str(e) or "cannot be found" in str(e).lower():
        raise ValueError(
            f"❌ ERROR: La tabla de tipo de cambio NO existe.\n"
            f"   Tabla: {tabla_tipo_cambio}\n"
            f"   Ejecuta el notebook 'str_gld_tipoCambioBCRP' primero.\n"
        )
    else:
        raise

print("=" * 60)

In [0]:
# ============================================================
# VALIDACIÓN #3: INTEGRIDAD REFERENCIAL (FAIL FAST)
# ============================================================

print("=" * 60)
print("VALIDACIÓN #3: INTEGRIDAD REFERENCIAL")
print("=" * 60)
print("Verificando que cod_producto y cod_tienda existen en dimensiones Silver")
print("-" * 60)

# 1. Extraer códigos únicos de la tabla de producción
codigos_producto_produccion = df_base.select("cod_producto").distinct()
codigos_tienda_produccion = df_base.select("cod_tienda").distinct()

total_productos = codigos_producto_produccion.count()
total_tiendas = codigos_tienda_produccion.count()

print(f"\nCódigos únicos en producción:")
print(f"  • cod_producto: {total_productos:,}")
print(f"  • cod_tienda: {total_tiendas:,}")

# 2. Leer tablas Silver de dimensiones
tabla_producto_silver = "adbsmartdatamanuelestrada.silver.base_producto_slv"
tabla_tienda_silver = "adbsmartdatamanuelestrada.silver.base_tiendas_slv"

try:
    df_producto_silver = spark.table(tabla_producto_silver).select("cod_producto").distinct()
    df_tienda_silver = spark.table(tabla_tienda_silver).select("cod_tienda").distinct()
    
    productos_disponibles = df_producto_silver.count()
    tiendas_disponibles = df_tienda_silver.count()
    
    print(f"\nCódigos únicos en Silver:")
    print(f"  • Productos disponibles: {productos_disponibles:,}")
    print(f"  • Tiendas disponibles: {tiendas_disponibles:,}")
    
except Exception as e:
    if "Table or view not found" in str(e) or "cannot be found" in str(e).lower():
        raise ValueError(
            f"❌ ERROR: Las tablas Silver de dimensiones NO existen.\n"
            f"   Tablas requeridas:\n"
            f"   - {tabla_producto_silver}\n"
            f"   - {tabla_tienda_silver}\n\n"
            f"   SOLUCIÓN:\n"
            f"   1. Verifica que Quality_Producto haya ejecutado exitosamente\n"
            f"   2. Verifica que Quality_Tienda haya ejecutado exitosamente\n"
            f"   3. Verifica las dependencias del job\n"
        )
    else:
        raise

# 3. VALIDAR cod_producto: buscar huérfanos (códigos en producción que NO existen en Silver)
print("\n" + "-" * 60)
print("Validando cod_producto...")
print("-" * 60)

productos_huerfanos = codigos_producto_produccion.join(
    df_producto_silver,
    on="cod_producto",
    how="left_anti"  # Solo registros de producción que NO matchean con Silver
)

num_productos_huerfanos = productos_huerfanos.count()

if num_productos_huerfanos > 0:
    print(f"\n❌ ERROR: Se encontraron {num_productos_huerfanos:,} cod_producto que NO existen en Silver")
    
    # Mostrar ejemplos
    print("\n🔍 Ejemplos de cod_producto huérfanos (no existen en dimensión):")
    ejemplos_producto = productos_huerfanos.limit(10).collect()
    for row in ejemplos_producto:
        print(f"  • {row['cod_producto']}")
    
    # Contar cuántos registros de producción se verían afectados
    registros_afectados_producto = df_base.join(
        productos_huerfanos,
        on="cod_producto",
        how="inner"
    ).count()
    
    print(f"\n⚠️  Impacto: {registros_afectados_producto:,} registros de producción tienen cod_producto inválido")
    
    raise ValueError(
        f"❌ VALIDACIÓN DE INTEGRIDAD REFERENCIAL FALLIDA\n"
        f"\n"
        f"   Se encontraron {num_productos_huerfanos:,} cod_producto en producción que NO existen en Silver.\n"
        f"   Esto afecta a {registros_afectados_producto:,} registros de producción.\n"
        f"\n"
        f"   CAUSAS POSIBLES:\n"
        f"   1. El archivo de productos (base_producto*) está desactualizado\n"
        f"   2. El archivo de producción contiene códigos nuevos aún no registrados\n"
        f"   3. Error en la carga de productos a Silver\n"
        f"\n"
        f"   SOLUCIÓN:\n"
        f"   1. Verifica que el archivo base_producto* esté actualizado en Landing\n"
        f"   2. Re-ejecuta Ingesta_Producto → Quality_Producto\n"
        f"   3. Verifica los códigos huérfanos mostrados arriba\n"
    )
else:
    print(f"✅ Todos los cod_producto ({total_productos:,}) existen en Silver")

# 4. VALIDAR cod_tienda: buscar huérfanos
print("\n" + "-" * 60)
print("Validando cod_tienda...")
print("-" * 60)

tiendas_huerfanas = codigos_tienda_produccion.join(
    df_tienda_silver,
    on="cod_tienda",
    how="left_anti"
)

num_tiendas_huerfanas = tiendas_huerfanas.count()

if num_tiendas_huerfanas > 0:
    print(f"\n❌ ERROR: Se encontraron {num_tiendas_huerfanas:,} cod_tienda que NO existen en Silver")
    
    # Mostrar ejemplos
    print("\n🔍 Ejemplos de cod_tienda huérfanos (no existen en dimensión):")
    ejemplos_tienda = tiendas_huerfanas.limit(10).collect()
    for row in ejemplos_tienda:
        print(f"  • {row['cod_tienda']}")
    
    # Contar cuántos registros de producción se verían afectados
    registros_afectados_tienda = df_base.join(
        tiendas_huerfanas,
        on="cod_tienda",
        how="inner"
    ).count()
    
    print(f"\n⚠️  Impacto: {registros_afectados_tienda:,} registros de producción tienen cod_tienda inválido")
    
    raise ValueError(
        f"❌ VALIDACIÓN DE INTEGRIDAD REFERENCIAL FALLIDA\n"
        f"\n"
        f"   Se encontraron {num_tiendas_huerfanas:,} cod_tienda en producción que NO existen en Silver.\n"
        f"   Esto afecta a {registros_afectados_tienda:,} registros de producción.\n"
        f"\n"
        f"   CAUSAS POSIBLES:\n"
        f"   1. El archivo de tiendas (base_tiendas*) está desactualizado\n"
        f"   2. El archivo de producción contiene códigos de tienda nuevos aún no registrados\n"
        f"   3. Error en la carga de tiendas a Silver\n"
        f"\n"
        f"   SOLUCIÓN:\n"
        f"   1. Verifica que el archivo base_tiendas* esté actualizado en Landing\n"
        f"   2. Re-ejecuta Ingesta_Tienda → Quality_Tienda\n"
        f"   3. Verifica los códigos huérfanos mostrados arriba\n"
    )
else:
    print(f"✅ Todos los cod_tienda ({total_tiendas:,}) existen en Silver")

print("\n" + "=" * 60)
print("✅ VALIDACIÓN DE INTEGRIDAD REFERENCIAL EXITOSA")
print("=" * 60)
print(f"  • {total_productos:,} cod_producto validados")
print(f"  • {total_tiendas:,} cod_tienda validados")
print(f"  • 0 códigos huérfanos encontrados")
print("\n  Se procederá con el procesamiento de calidad de datos.")
print("=" * 60)

In [0]:
# ============================================================
# EDA: ANÁLISIS DE CALIDAD DE DATOS ORIGINALES (df_base)
# ============================================================

print("\n" + "=" * 70)
print("1. ANÁLISIS DE VALORES NULOS (ORIGINALES DE BRONZE)")
print("=" * 70)

total_registros = df_base.count()
print(f"\nTotal de registros: {total_registros:,}\n")

# Analizar nulos por columna
print(f"{'Columna':<25} {'Nulos':>10} {'% Nulos':>10}")
print("-" * 50)

nulos_por_columna = {}
for column in df_base.columns:
    null_count = df_base.filter(F.col(column).isNull() | (F.col(column) == "")).count()
    porc_null = (null_count / total_registros * 100) if total_registros > 0 else 0
    nulos_por_columna[column] = null_count
    print(f"{column:<25} {null_count:>10,} {porc_null:>9.2f}%")

print("\n" + "=" * 70)
print("2. ANÁLISIS DE DUPLICADOS EXACTOS")
print("=" * 70)

# Contar duplicados exactos
registros_unicos = df_base.dropDuplicates().count()
duplicados = total_registros - registros_unicos
porc_duplicados = (duplicados / total_registros * 100) if total_registros > 0 else 0

print(f"\nRegistros totales:        {total_registros:,}")
print(f"Registros únicos:         {registros_unicos:,}")
print(f"Registros duplicados:     {duplicados:,} ({porc_duplicados:.2f}%)")

print("\n" + "=" * 70)
print("3. RESUMEN DE CALIDAD DE DATOS")
print("=" * 70)

total_nulos = sum(nulos_por_columna.values())
print(f"\nTotal de valores nulos:   {total_nulos:,}")
print(f"Campos con nulos:         {sum(1 for v in nulos_por_columna.values() if v > 0)}/{len(nulos_por_columna)}")
print(f"Duplicados exactos:       {duplicados:,}")

In [0]:
# ============================================================
# DEBUG: EXPORTAR REGISTROS PROBLEMÁTICOS A CSV
# ============================================================
from datetime import datetime

print("\n" + "=" * 70)
print("GENERANDO ARCHIVO DE DEBUGGING")
print("=" * 70)

# 1. IDENTIFICAR REGISTROS CON NULOS ORIGINALES
df_con_nulos_originales = df_base.withColumn(
    "problema", F.lit("NULO_ORIGINAL")
).withColumn(
    "detalle_problema",
    F.concat_ws(", ",
        F.when(F.col("fecha_orden").isNull() | (F.col("fecha_orden") == ""), F.lit("fecha_orden")),
        F.when(F.col("cod_venta").isNull() | (F.col("cod_venta") == ""), F.lit("cod_venta")),
        F.when(F.col("cod_producto").isNull() | (F.col("cod_producto") == ""), F.lit("cod_producto")),
        F.when(F.col("cod_cliente").isNull() | (F.col("cod_cliente") == ""), F.lit("cod_cliente")),
        F.when(F.col("cantidad").isNull() | (F.col("cantidad") == ""), F.lit("cantidad")),
        F.when(F.col("precio_unit").isNull() | (F.col("precio_unit") == ""), F.lit("precio_unit")),
        F.when(F.col("coste_unit").isNull() | (F.col("coste_unit") == ""), F.lit("coste_unit")),
        F.when(F.col("porc_iva").isNull() | (F.col("porc_iva") == ""), F.lit("porc_iva"))
    )
).filter(
    F.col("fecha_orden").isNull() | (F.col("fecha_orden") == "") |
    F.col("cod_venta").isNull() | (F.col("cod_venta") == "") |
    F.col("cod_producto").isNull() | (F.col("cod_producto") == "") |
    F.col("cod_cliente").isNull() | (F.col("cod_cliente") == "") |
    F.col("cantidad").isNull() | (F.col("cantidad") == "") |
    F.col("precio_unit").isNull() | (F.col("precio_unit") == "") |
    F.col("coste_unit").isNull() | (F.col("coste_unit") == "") |
    F.col("porc_iva").isNull() | (F.col("porc_iva") == "")
)

nulos_count = df_con_nulos_originales.count()
print(f"\nRegistros con nulos originales: {nulos_count:,}")

# 2. IDENTIFICAR DUPLICADOS EXACTOS
window_spec_debug = Window.partitionBy(*df_base.columns)
df_duplicados_debug = df_base.withColumn(
    "duplicate_count", F.count("*").over(window_spec_debug)
).filter(
    F.col("duplicate_count") > 1
).withColumn(
    "problema", F.lit("DUPLICADO_EXACTO")
).withColumn(
    "detalle_problema", 
    F.concat(F.lit("Aparece "), F.col("duplicate_count").cast("string"), F.lit(" veces"))
).drop("duplicate_count")

duplicados_count = df_duplicados_debug.count()
print(f"Registros duplicados: {duplicados_count:,}")

# 3. IDENTIFICAR INCONSISTENCIAS DE CONVERSIÓN
df_nivel1_temp = df_base.withColumn(
    "fecha_orden_conv", F.to_date(F.col("fecha_orden"), "d/MM/yyyy")
).withColumn(
    "cantidad_conv", F.col("cantidad").cast(IntegerType())
).withColumn(
    "precio_unit_conv", F.col("precio_unit").cast(DoubleType())
).withColumn(
    "coste_unit_conv", F.col("coste_unit").cast(DoubleType())
).withColumn(
    "porc_iva_conv", F.col("porc_iva").cast(IntegerType())
)

df_inconsistencias = df_nivel1_temp.filter(
    ((F.col("fecha_orden").isNotNull()) & (F.col("fecha_orden") != "") & F.col("fecha_orden_conv").isNull()) |
    ((F.col("cantidad").isNotNull()) & (F.col("cantidad") != "") & F.col("cantidad_conv").isNull()) |
    ((F.col("precio_unit").isNotNull()) & (F.col("precio_unit") != "") & F.col("precio_unit_conv").isNull()) |
    ((F.col("coste_unit").isNotNull()) & (F.col("coste_unit") != "") & F.col("coste_unit_conv").isNull()) |
    ((F.col("porc_iva").isNotNull()) & (F.col("porc_iva") != "") & F.col("porc_iva_conv").isNull())
).withColumn(
    "problema", F.lit("INCONSISTENCIA_CONVERSION")
).withColumn(
    "detalle_problema",
    F.concat_ws(", ",
        F.when((F.col("fecha_orden").isNotNull()) & (F.col("fecha_orden_conv").isNull()), 
               F.lit("fecha_orden")),
        F.when((F.col("cantidad").isNotNull()) & (F.col("cantidad") != "") & (F.col("cantidad_conv").isNull()), 
               F.lit("cantidad")),
        F.when((F.col("precio_unit").isNotNull()) & (F.col("precio_unit") != "") & (F.col("precio_unit_conv").isNull()), 
               F.lit("precio_unit")),
        F.when((F.col("coste_unit").isNotNull()) & (F.col("coste_unit") != "") & (F.col("coste_unit_conv").isNull()), 
               F.lit("coste_unit")),
        F.when((F.col("porc_iva").isNotNull()) & (F.col("porc_iva") != "") & (F.col("porc_iva_conv").isNull()), 
               F.lit("porc_iva"))
    )
).drop("fecha_orden_conv", "cantidad_conv", "precio_unit_conv", "coste_unit_conv", "porc_iva_conv")

inconsistencias_count = df_inconsistencias.count()
print(f"Registros con inconsistencias de conversión: {inconsistencias_count:,}")

# 4. UNIR TODOS LOS REGISTROS PROBLEMÁTICOS
df_problemas_todos = df_con_nulos_originales.unionByName(
    df_duplicados_debug, allowMissingColumns=True
).unionByName(
    df_inconsistencias, allowMissingColumns=True
)

total_problemas = df_problemas_todos.count()
print(f"\nTotal de registros problemáticos: {total_problemas:,}")

# 5. EXPORTAR A CSV ÚNICO (NO CARPETA) - USANDO PANDAS
if total_problemas > 0:
    # Obtener el mes del archivo (fecha_corte)
    mes_archivo = df_base.select(
        F.concat(
            F.year(F.to_date(F.col("fecha_orden"), "d/MM/yyyy")),
            F.lpad(F.month(F.to_date(F.col("fecha_orden"), "d/MM/yyyy")), 2, "0")
        ).alias("mes")
    ).filter(F.col("mes").isNotNull()).limit(1).collect()
    
    fecha_corte = mes_archivo[0]["mes"] if mes_archivo else "unknown"
    
    # Nombre del archivo: Archivo_Debug_(fecha_corte).csv
    csv_filename = f"Archivo_Debug_{fecha_corte}.csv"
    storage_account_debug = "adlssmartdatamanuel95"
    ruta_debug_adls = f"abfss://raw@{storage_account_debug}.dfs.core.windows.net/debug/{csv_filename}"
    
    # Reordenar columnas: problema y detalle_problema primero
    columnas_base = [col for col in df_base.columns]
    df_export = df_problemas_todos.select(
        F.col("problema"),
        F.col("detalle_problema"),
        *columnas_base
    )
    
    # Convertir a pandas y escribir CSV único (NO carpeta)
    print(f"\nConvirtiendo a pandas...")
    pdf_export = df_export.toPandas()
    
    import os
    temp_path = f"/tmp/{csv_filename}"
    print(f"Escribiendo archivo CSV único: {csv_filename}...")
    pdf_export.to_csv(temp_path, index=False)
    
    # Copiar desde driver local a ADLS Gen2
    dbutils.fs.cp(f"file://{temp_path}", ruta_debug_adls)
    os.remove(temp_path)
    
    print(f"\n✅ Archivo CSV único exportado exitosamente a ADLS:")
    print(f"   Nombre: {csv_filename}")
    print(f"   Ruta: {ruta_debug_adls}")
    print(f"   Registros exportados: {total_problemas:,}")
    print(f"   - Nulos originales: {nulos_count:,}")
    print(f"   - Duplicados: {duplicados_count:,}")
    print(f"   - Inconsistencias: {inconsistencias_count:,}")
else:
    print("\n✅ No se encontraron registros problemáticos.")
    print("   No se generó archivo de debugging.")

print("=" * 70)

In [0]:
# NIVEL 1: Conversión de tipos de datos
registros_recibidos = df_base.count()

# Contar nulos ANTES de la conversión (nulos originales)
nulos_fecha_antes = df_base.filter(F.col("fecha_orden").isNull() | (F.col("fecha_orden") == "")).count()
nulos_cantidad_antes = df_base.filter(F.col("cantidad").isNull() | (F.col("cantidad") == "")).count()
nulos_precio_antes = df_base.filter(F.col("precio_unit").isNull() | (F.col("precio_unit") == "")).count()
nulos_coste_antes = df_base.filter(F.col("coste_unit").isNull() | (F.col("coste_unit") == "")).count()
nulos_porc_iva_antes = df_base.filter(F.col("porc_iva").isNull() | (F.col("porc_iva") == "")).count()

# Convertir tipos de datos
# Primero parseamos la fecha con el formato original d/MM/yyyy y luego la convertimos a formato yyyy-MM-dd
df_nivel1 = df_base.withColumn("fecha_orden", F.to_date(F.col("fecha_orden"), "d/MM/yyyy")) \
    .withColumn("cod_venta", F.col("cod_venta").cast("string")) \
    .withColumn("cod_producto", F.col("cod_producto").cast("string")) \
    .withColumn("cod_cliente", F.col("cod_cliente").cast("string")) \
    .withColumn("cantidad", F.col("cantidad").cast(IntegerType())) \
    .withColumn("precio_unit", F.col("precio_unit").cast(DoubleType())) \
    .withColumn("coste_unit", F.col("coste_unit").cast(DoubleType())) \
    .withColumn("porc_iva", F.col("porc_iva").cast(IntegerType())) \
    .withColumn("data_ingestion_ts", F.to_timestamp(F.col("data_ingestion_ts")))

# Contar nulos DESPUÉS de la conversión
nulos_fecha_despues = df_nivel1.filter(F.col("fecha_orden").isNull()).count()
nulos_cantidad_despues = df_nivel1.filter(F.col("cantidad").isNull()).count()
nulos_precio_despues = df_nivel1.filter(F.col("precio_unit").isNull()).count()
nulos_coste_despues = df_nivel1.filter(F.col("coste_unit").isNull()).count()
nulos_porc_iva_despues = df_nivel1.filter(F.col("porc_iva").isNull()).count()

# INCONSISTENCIAS = valores que se convirtieron a NULL por conversión fallida
incons_fecha = nulos_fecha_despues - nulos_fecha_antes
incons_cantidad = nulos_cantidad_despues - nulos_cantidad_antes
incons_precio = nulos_precio_despues - nulos_precio_antes
incons_coste = nulos_coste_despues - nulos_coste_antes
incons_porc_iva = nulos_porc_iva_despues - nulos_porc_iva_antes

# Total de registros con al menos una inconsistencia de conversión
registros_con_inconsistencias = max(incons_fecha, incons_cantidad, incons_precio, incons_coste, incons_porc_iva)

registros_salida = df_nivel1.count()
registros_eliminados = 0  # En este nivel no eliminamos, solo convertimos
porc_pasan = (registros_salida / registros_recibidos * 100) if registros_recibidos > 0 else 0

print("=" * 60)
print("FUNNEL NIVEL 1: CONVERSIÓN DE TIPOS DE DATOS")
print("=" * 60)
print(f"Registros recibidos:              {registros_recibidos:,}")
print(f"Registros con inconsistencias:    {registros_con_inconsistencias:,}")
print(f"  - fecha_orden (null):           {incons_fecha:,}")
print(f"  - cantidad (null):              {incons_cantidad:,}")
print(f"  - precio_unit (null):           {incons_precio:,}")
print(f"  - coste_unit (null):            {incons_coste:,}")
print(f"  - porc_iva (null):              {incons_porc_iva:,}")
print(f"Registros eliminados:             {registros_eliminados:,}")
print(f"Registros que pasan a producción: {registros_salida:,}")
print(f"% registros que pasan:            {porc_pasan:.2f}%")
print("=" * 60)

In [0]:
# NIVEL 2: Eliminación de duplicados exactos
registros_recibidos = df_nivel1.count()

# Agregar columna de conteo de duplicados
window_spec = Window.partitionBy(*df_nivel1.columns)
df_con_conteo = df_nivel1.withColumn("duplicate_count", F.count("*").over(window_spec))

# Filtrar registros duplicados (aparecen más de una vez)
df_duplicados = df_con_conteo.filter(F.col("duplicate_count") > 1).drop("duplicate_count")

# Eliminar duplicados exactos (registros que tienen todos los valores idénticos)
df_nivel2 = df_nivel1.dropDuplicates()

registros_salida = df_nivel2.count()
registros_eliminados = registros_recibidos - registros_salida
porc_pasan = (registros_salida / registros_recibidos * 100) if registros_recibidos > 0 else 0

print("=" * 60)
print("FUNNEL NIVEL 2: ELIMINACIÓN DE DUPLICADOS EXACTOS")
print("=" * 60)
print(f"Registros recibidos:              {registros_recibidos:,}")
print(f"Registros eliminados:             {registros_eliminados:,}")
print(f"Registros que pasan a producción: {registros_salida:,}")
print(f"% registros que pasan:            {porc_pasan:.2f}%")
print("=" * 60)

if registros_eliminados > 0:
    print("\nMUESTRA DE 3 REGISTROS DUPLICADOS (ELIMINADOS):")
    display(df_duplicados.limit(4))

In [0]:
# NIVEL 3: Validación de nulos en campos críticos (cod_producto, precio_unit, coste_unit, cantidad)
registros_recibidos = df_nivel2.count()

# Identificar registros con nulos ANTES de eliminarlos (para trazabilidad)
df_con_nulos = df_nivel2.filter(
    F.col("cod_producto").isNull() |
    F.col("precio_unit").isNull() |
    F.col("coste_unit").isNull() |
    F.col("cantidad").isNull()
)

# Eliminar registros con nulos en cod_producto, precio_unit, coste_unit o cantidad
df_nivel3 = df_nivel2.filter(
    F.col("cod_producto").isNotNull() &
    F.col("precio_unit").isNotNull() &
    F.col("coste_unit").isNotNull() &
    F.col("cantidad").isNotNull()
)

registros_salida = df_nivel3.count()
registros_eliminados = registros_recibidos - registros_salida
porc_pasan = (registros_salida / registros_recibidos * 100) if registros_recibidos > 0 else 0

print("=" * 60)
print("FUNNEL NIVEL 3: VALIDACIÓN DE NULOS EN CAMPOS CRÍTICOS")
print("=" * 60)
print(f"Registros recibidos:              {registros_recibidos:,}")
print(f"Registros eliminados:             {registros_eliminados:,}")
print(f"Registros que pasan a producción: {registros_salida:,}")
print(f"% registros que pasan:            {porc_pasan:.2f}%")
print("=" * 60)

if registros_eliminados > 0:
    print("\nMUESTRA DE 3 REGISTROS CON NULOS EN CAMPOS CRÍTICOS (ELIMINADOS):")
    display(df_con_nulos.limit(3))

In [0]:
# REGLA DE NEGOCIO 1: Llenado de nulos en fecha_orden
# Si hay fechas nulas, se rellenan con el último día del mes más frecuente

# Obtener el mes y año de las fechas no nulas
df_con_mes = df_nivel3.withColumn("year_month", 
    F.when(F.col("fecha_orden").isNotNull(), 
           F.concat(F.year("fecha_orden"), F.lit("-"), F.lpad(F.month("fecha_orden"), 2, "0")))
)

# Obtener el mes más frecuente en los datos
mes_frecuente = df_con_mes.filter(F.col("year_month").isNotNull()) \
    .groupBy("year_month").count() \
    .orderBy(F.desc("count")) \
    .limit(1) \
    .collect()

if mes_frecuente:
    year_month = mes_frecuente[0]["year_month"]
    year, month = year_month.split("-")
    year, month = int(year), int(month)
    
    # Calcular el PRIMER día del mes (consistente con truncamiento en Analityc_layer)
    fecha_relleno = date(year, month, 1)
    
    # Identificar registros con fecha_orden nula ANTES de rellenar (para trazabilidad)
    df_fechas_nulas = df_nivel3.filter(F.col("fecha_orden").isNull())
    registros_con_fecha_nula = df_fechas_nulas.count()
    
    print("=" * 60)
    print("REGLA DE NEGOCIO 1: LLENADO DE FECHAS NULAS")
    print("=" * 60)
    print(f"Registros con fecha nula:       {registros_con_fecha_nula:,}")
    print(f"Mes más frecuente en los datos: {year_month}")
    print(f"Fecha de relleno (primer día):  {fecha_relleno}")
    print("=" * 60)
    
    # Rellenar nulos con el PRIMER día del mes
    df_nivel4 = df_nivel3.withColumn("fecha_orden",
        F.when(F.col("fecha_orden").isNull(), F.lit(fecha_relleno)).otherwise(F.col("fecha_orden"))
    )
    
    if registros_con_fecha_nula > 0:
        # Mostrar muestra de registros que fueron rellenados
        df_rellenados = df_nivel4.join(
            df_fechas_nulas.select("cod_venta"),
            "cod_venta",
            "inner"
        )
        print("\nMUESTRA DE 3 REGISTROS CON FECHA SUBSANADA:")
        display(df_rellenados.limit(3))
else:
    # Si no hay fechas válidas, mantener el dataframe sin cambios
    print("No se encontraron fechas válidas para determinar el mes más frecuente.")
    df_nivel4 = df_nivel3

In [0]:
# REGLA DE NEGOCIO 2: Ninguna cantidad debe ser 0 o menor a 0
# Eliminar registros donde cantidad <= 0

registros_recibidos = df_nivel4.count()

# Filtrar registros con cantidad > 0
df_nivel5 = df_nivel4.filter(F.col("cantidad") > 0)

registros_salida = df_nivel5.count()
registros_eliminados = registros_recibidos - registros_salida

print("=" * 60)
print("REGLA DE NEGOCIO 2: VALIDACIÓN CANTIDAD > 0")
print("=" * 60)
print(f"Registros recibidos:     {registros_recibidos:,}")
print(f"Registros eliminados:    {registros_eliminados:,}")
print(f"Registros que pasan:     {registros_salida:,}")
print("=" * 60)

display(df_nivel5.limit(5))

In [0]:
# REGLA DE NEGOCIO 3: Ningún coste_unit debe ser 0 o menor a 0
# Eliminar registros donde coste_unit <= 0

registros_recibidos = df_nivel5.count()

# Filtrar registros con coste_unit > 0
df_nivel6 = df_nivel5.filter(F.col("coste_unit") > 0)

registros_salida = df_nivel6.count()
registros_eliminados = registros_recibidos - registros_salida

print("=" * 60)
print("REGLA DE NEGOCIO 3: VALIDACIÓN COSTE_UNIT > 0")
print("=" * 60)
print(f"Registros recibidos:     {registros_recibidos:,}")
print(f"Registros eliminados:    {registros_eliminados:,}")
print(f"Registros que pasan:     {registros_salida:,}")
print("=" * 60)

display(df_nivel6.limit(5))

In [0]:
# REGLA DE NEGOCIO 4: Ningún precio_unit debe ser 0 o menor a 0
# Eliminar registros donde precio_unit <= 0

registros_recibidos = df_nivel6.count()

# Filtrar registros con precio_unit > 0
df_nivel7 = df_nivel6.filter(F.col("precio_unit") > 0)

registros_salida = df_nivel7.count()
registros_eliminados = registros_recibidos - registros_salida

print("=" * 60)
print("REGLA DE NEGOCIO 4: VALIDACIÓN PRECIO_UNIT > 0")
print("=" * 60)
print(f"Registros recibidos:     {registros_recibidos:,}")
print(f"Registros eliminados:    {registros_eliminados:,}")
print(f"Registros que pasan:     {registros_salida:,}")
print("=" * 60)

print(" DATAFRAME FINAL (df_nivel7)")
display(df_nivel7.limit(5))

In [0]:
# Guardar el DataFrame final como tabla Silver en Unity Catalog
table_name = "adbsmartdatamanuelestrada.silver.quality_layer_slv"

print("=" * 60)
print("ESCRITURA A TABLA SILVER")
print("=" * 60)

registros_a_insertar = df_nivel7.count()
print(f"\nRegistros a insertar: {registros_a_insertar:,}")

# Escribir en modo append (la validación de duplicados ya se hizo al inicio)
df_nivel7.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(table_name)

print("\n" + "=" * 60)
print("✅ INSERCIÓN EXITOSA")
print("=" * 60)
print(f"Tabla: {table_name}")
print(f"Registros insertados: {registros_a_insertar:,}")
print(f"Total registros en tabla: {spark.table(table_name).count():,}")
print("=" * 60)